# Task 2 — CoT + self-consistency với Qwen2.5-Coder-7B-Instruct (Kaggle)

Cùng prompt giảng viên với `kaggle/task2_cot.ipynb`, nhưng mỗi bài được chấm **5 lần** (temperature 0.7, top_p 0.95) rồi bỏ phiếu: giữ nhãn có ≥ 50% phiếu (`configs/task2/cot_self_consistency.yaml`, script `scripts/run_task2_cot_sc.py`).
5 câu trả lời của một bài được sinh trong **một** lần gọi `generate` (`num_return_sequences=5`), nên chậm hơn zero-shot khoảng 1.3–2 lần chứ không phải 5 lần.

1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. `DRIVE_ID` (không commit) hoặc secret `PUBLIC_TEST_DRIVE_ID`; tuỳ chọn `GITHUB_TOKEN`, `HF_TOKEN`. Sửa `BRANCH`.
3. Run All.

**So sánh với zero-shot:** `BASELINE = True` chạy thêm zero-shot CoT trên 32 gold (≈30 phút) trước; script SC in bảng so sánh và ghi `compare.json`.

**360 bài public quá 12 giờ?** Chia shard qua nhiều phiên:
- Phiên 1: `PUBLIC_SHARD = '1/2'` → Save Version. Output `/kaggle/working/outputs` chứa cache (`raw_outputs.jsonl`).
- Phiên 2: thêm output phiên 1 làm Input (Add Input → Notebook Output), đặt `RESUME_FROM = ['/kaggle/input/<tên>/outputs']`, `PUBLIC_SHARD = '2/2'`.
- Phiên cuối: `RESUME_FROM` = output các phiên trước, `PUBLIC_SHARD = ''` → mọi bài lấy từ cache, ghi `task2_predictions.json` đủ 360 bài.

**Silver (tập sinh, để chỉnh prompt):** đặt `SILVER = 'silver'` (462 bài) hoặc `'heldout'` (65 bài, 4 đề chưa thấy) và `SILVER_DRIVE_ID` (hoặc secret `SILVER_DRIVE_ID`, file `generate-dataset.zip`). Notebook tải về `data/synth/v1/generate-dataset/` bằng `scripts/fetch_synth_data.py` rồi chấm thêm tập đó (thư mục kết quả có hậu tố `_silver` / `_silver_heldout`). Chỉnh prompt theo số trên silver, không theo gold.

**Cấu hình (`EXPERIMENT`)** — mọi cấu hình đều lấy `Lỗi biên dịch` từ compile_log (warning / thiếu `main` không tính) và map tên mục ("Vòng lặp") về tên nhãn:
- `cot_zero_shot`: chỉ prompt giảng viên. `cot_fewshot`: thêm 3 bài đã chấm giống nhất (gold trước, silver bù; không bao giờ lấy bài public) — cần `SILVER_DRIVE_ID`.
- `cot_self_consistency` / `cot_fewshot_sc`: như trên + 5 lời chấm, bỏ phiếu ≥ 50%.

**Output** (`/kaggle/working`): `task2_predictions.json` (khi không chia shard), `outputs/<run>/` gồm `report.html` (lọc "Phiếu chia rẽ", xem 5 lời chấm và số phiếu từng nhãn), `debug.jsonl`, `run.log`, `raw_outputs.jsonl`, `usage.jsonl`, `metrics.json`, `compare.json`; `task2_cot_sc_bundle/` để chạy lại sau.
*Lưu ý:* `debug.jsonl`/`report.html` chứa code bài nộp để debug, nên giữ notebook version ở chế độ private.


In [ ]:
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2-zeroshot'# nhánh chứa scripts/run_task2_cot_sc.py
DRIVE_ID = ''             # paste the Google Drive file id here on Kaggle (do not commit it: the repo is public)
WORK = '/kaggle/tmp'      # student data + repo live here
OUT = '/kaggle/working/outputs'  # run folders + cache, kept with the notebook version so a later session can resume
EXPERIMENT = 'cot_fewshot_sc_en'  # prompt tiếng Anh: 'cot_self_consistency_en' (zero-shot + SC) | 'cot_fewshot_sc_en' (few-shot + SC; needs SILVER_DRIVE_ID); bỏ '_en' = tiếng Việt
BASELINE_EXPERIMENT = 'cot_zero_shot_en'  # zero-shot cùng ngôn ngữ prompt, để so sánh
COMPILE_SOURCE = 'self'   # 'dataset' if DRIVE_ID points to the compiled zip (compile_log already filled)
BASELINE = True           # also run zero-shot CoT on the 32 gold first, for the comparison table
GOLD = True               # score SC on the 32 gold
PREDICT = True            # label the 360 public samples
PUBLIC_SHARD = ''         # e.g. '1/2' then '2/2' over two sessions; '' = all (cached ones are skipped)
RESUME_FROM = []          # e.g. ['/kaggle/input/<previous-version>/outputs'] to reuse earlier sessions' cache
LIMIT = None              # e.g. 4 for a quick check on the first gold samples (no public run)
BACKEND = 'vllm'          # 'vllm': continuous batching, ~5-10x faster, KV cache reserved up front (no random OOM) | 'hf': transformers, one sample at a time
VLLM_VERSION = ''         # '' = newest wheel for this Python (Kaggle is on 3.13; vLLM 0.9.x has no 3.13 wheel); or pin e.g. '0.11.0'
LOAD_IN_4BIT = False
SAVE_WEIGHTS = False      # True: copy the ~15 GB fp16 weights into the bundle
SILVER_DRIVE_ID = ''      # Drive file id of generate-dataset.zip (or secret SILVER_DRIVE_ID); do not commit
SILVER = ''               # '' = off | 'silver' (462 synthetic, prompt tuning) | 'heldout' (65, 4 unseen exams)
SILVER_LIMIT = None       # e.g. 100: only the first N silver samples


In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
os.makedirs(WORK, exist_ok=True)
REPO_DIR = f'{WORK}/repo'
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed — check REPO/BRANCH/GITHUB_TOKEN') from None  # never print the token URL
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)  # drop token from .git/config
os.chdir(REPO_DIR)
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q gdown 'transformers>=4.51,<5' accelerate safetensors scikit-learn pyyaml bitsandbytes
if BACKEND == 'vllm':
    VLLM_PKG = f'vllm=={VLLM_VERSION}' if VLLM_VERSION else 'vllm'
    !pip install -q {VLLM_PKG}  # brings its own torch: takes a few minutes
    import subprocess, sys
    r = subprocess.run([sys.executable, '-c', 'import vllm, torch; print(vllm.__version__, torch.__version__)'],
                       capture_output=True, text=True)
    if r.returncode:  # pip errors are easy to miss with -q: say it and fall back instead of failing later
        print(r.stderr[-3000:])
        print(f'!!! {VLLM_PKG} không import được (xem lỗi pip ở trên) -> chuyển BACKEND = "hf"')
        BACKEND = 'hf'
    else:
        print('vllm, torch:', r.stdout.strip())
!g++ --version


In [ ]:
import gdown
drive_id = DRIVE_ID or secret('PUBLIC_TEST_DRIVE_ID')
assert drive_id, 'Set DRIVE_ID in the config cell (or add the PUBLIC_TEST_DRIVE_ID secret)'
os.makedirs(f'{WORK}/download', exist_ok=True)
path = gdown.download(id=drive_id, output=f'{WORK}/download/data.zip', quiet=False)
assert path is not None, 'gdown failed — check PUBLIC_TEST_DRIVE_ID and sharing permissions'
!python scripts/unpack_data.py {WORK}/download --dest data/raw
!ls data/raw

In [ ]:
# Synthetic silver set -> data/synth/v1/generate-dataset/ (+ heldout/), only when SILVER is set
if SILVER or 'fewshot' in EXPERIMENT:  # few-shot demos also come from the silver set
    os.environ['SILVER_DRIVE_ID'] = SILVER_DRIVE_ID or secret('SILVER_DRIVE_ID')
    assert os.environ['SILVER_DRIVE_ID'], 'Set SILVER_DRIVE_ID in the config cell (or add the SILVER_DRIVE_ID secret)'
    !python scripts/fetch_synth_data.py


In [ ]:
# Resume: merge earlier sessions' run folders into OUT (raw_outputs.jsonl / usage.jsonl are concatenated: the cache
# keeps, per sample, any line whose key matches the current prompt + generation settings)
import glob, os, shutil
os.makedirs(OUT, exist_ok=True)
for src in RESUME_FROM:
    for run in glob.glob(f'{src}/*/'):
        dst = f'{OUT}/{os.path.basename(run.rstrip("/"))}'
        os.makedirs(dst, exist_ok=True)
        for name in ('raw_outputs.jsonl', 'usage.jsonl'):
            if os.path.exists(f'{run}/{name}'):
                with open(f'{dst}/{name}', 'a', encoding='utf-8') as out, open(f'{run}/{name}', encoding='utf-8') as inp:
                    data = inp.read()
                    out.write(data if data.endswith('\n') or not data else data + '\n')
        print('resumed', run, '->', dst)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import yaml
NAME = yaml.safe_load(open(f'configs/task2/{EXPERIMENT}.yaml'))['name']  # run folder prefix
run_cfg = yaml.safe_load(open('configs/task2/run_cot.yaml'))
run_cfg['hf']['load_in_4bit'] = LOAD_IN_4BIT  # hf only; vllm runs fp16 (vllm: section of run_cot.yaml)
yaml.safe_dump(run_cfg, open(f'{WORK}/run_cot.yaml', 'w'), allow_unicode=True)
COMMON = ['--run-config', f'{WORK}/run_cot.yaml', '--backend', BACKEND, '--compile-source', COMPILE_SOURCE,
          '--output-dir', OUT] + (['--limit', str(LIMIT)] if LIMIT else [])
BASE_CMD = ['python', 'scripts/run_task2_cot.py', '--experiment', BASELINE_EXPERIMENT, *COMMON]
SC_CMD = ['python', 'scripts/run_task2_cot_sc.py', '--experiment', EXPERIMENT, '--baseline', BASELINE_EXPERIMENT, *COMMON, '--export-bundle', '/kaggle/working/task2_cot_sc_bundle']
if not GOLD:
    SC_CMD += ['--skip-gold']
if PREDICT and not LIMIT:
    SC_CMD += ['--predict'] + (['--public-shard', PUBLIC_SHARD] if PUBLIC_SHARD else [])
if SAVE_WEIGHTS:
    SC_CMD += ['--save-weights']
print(' '.join(BASE_CMD) if BASELINE else '(no baseline)')
print(' '.join(SC_CMD))


In [ ]:
# Zero-shot baseline on gold (cached after the first time)
if BASELINE and GOLD:
    !{' '.join(BASE_CMD)}


In [ ]:
# Self-consistency: one line per sample with the 5 answers' token counts. Everything is also in run.log.
!{' '.join(SC_CMD)}


In [ ]:
# Score the silver set (prompt tuning). Run folder suffix: _silver / _silver_heldout
if SILVER:
    SILVER_FLAG = '--silver' if SILVER == 'silver' else '--silver-heldout'
    SBASE = ['--run-config', f'{WORK}/run_cot.yaml', '--backend', BACKEND, '--compile-source', COMPILE_SOURCE,
             '--output-dir', OUT, SILVER_FLAG] + (['--limit', str(SILVER_LIMIT)] if SILVER_LIMIT else [])
    if BASELINE:
        S0 = ' '.join(['python', 'scripts/run_task2_cot.py', '--experiment', BASELINE_EXPERIMENT, *SBASE])
        !{S0}
    S1 = ' '.join(['python', 'scripts/run_task2_cot_sc.py', '--experiment', EXPERIMENT, '--baseline', BASELINE_EXPERIMENT, *SBASE])
    print(S1)
    !{S1}


In [ ]:
import json
from src.task2.evaluator import validate_predictions
sc_gold = [d for d in glob.glob(f'{OUT}/{NAME}__{BACKEND}__*') if not d.endswith('_predict')]
sc_pred = glob.glob(f'{OUT}/{NAME}__{BACKEND}__*_predict')
if sc_gold and os.path.exists(f'{sc_gold[0]}/compare.json'):
    print(open(f'{sc_gold[0]}/compare.json').read()[:3000])
if sc_pred and not os.path.exists(f'{sc_pred[0]}/predictions.json'):
    print('CHƯA có predictions.json: lần chạy public dừng giữa chừng. Xem run.log / report.html (lọc OOM), '
          'rồi chạy lại cell self-consistency: các bài đã xong lấy từ cache trong OUT.')
elif sc_pred:
    sub = json.load(open(f'{sc_pred[0]}/predictions.json'))
    errs = validate_predictions(sub)
    print('public predictions:', len(sub), 'errors:', errs)
    if PUBLIC_SHARD:
        print(f'shard {PUBLIC_SHARD}: chưa đủ 360 bài; chạy phiên cuối với PUBLIC_SHARD = "" để gộp')
    elif len(sub) == 360 and not errs:
        shutil.copy(f'{sc_pred[0]}/predictions.json', '/kaggle/working/task2_predictions.json')
        print('-> /kaggle/working/task2_predictions.json')


In [ ]:
# Per-label view of the SC gold run vs zero-shot (same numbers as compare.json / report.html)
import pandas as pd
if sc_gold and os.path.exists(f'{sc_gold[0]}/compare.json'):
    c = json.load(open(f'{sc_gold[0]}/compare.json'))
    df = pd.DataFrame({k: {'self-consistency': v['run'], 'zero-shot': v['baseline']}
                       for k, v in {**c['overall'], **c['per_label_f1']}.items()}).T
    display(df)


In [ ]:
# Interactive debug report inline (filter "Phiếu chia rẽ" to see where the 5 answers disagree)
import html as _html
from IPython.display import HTML, display
target = sc_gold[0] if sc_gold else (sc_pred[0] if sc_pred else None)
if target:
    page = open(f'{target}/report.html', encoding='utf-8').read()
    display(HTML(f'<iframe srcdoc="{_html.escape(page)}" style="width:100%;height:900px;border:0"></iframe>'))
